# Distillation: How Do You Compress a Large Model Into a Small One?

> **Background**: GPT-4 is strong but expensive and slow. Can we distill its capabilities into a 7B model so the 7B model approaches GPT-4 quality?
>
> Goal for this part: **understand three mainstream distillation methods for LLMs, and the end-to-end workflow of distilling from GPT-4 to a smaller model.**

One-sentence core idea:
**Distillation = a large model (Teacher) teaches a small model (Student). The student learns not only the final answers, but the teacher's "way of thinking".**

Note: for a deeper dive into On-Policy Distillation (OPD), see Part 23.


In [ ]:
import numpy as np

np.random.seed(42)

### 1. The essence: learn "how to judge" not only "what the answer is"

```
Standard SFT:
  Teacher output: "Paris"
  Student learns: input "Capital of France?" -> output "Paris"
  Problem: it learns the answer, but not the reasoning or uncertainty

Distillation:
  Teacher output: a probability distribution over tokens
    [Paris:0.90, London:0.05, Berlin:0.03, ...]
  Student learns: not only output "Paris", but match the whole distribution
  Benefit: Student learns teacher's calibration and preferences
```

Why is a distribution more valuable than a single answer?

"Paris 90%, London 5%, Berlin 3%" contains extra information:
1. London/Berlin are plausible but less likely ("dark knowledge")
2. many other options are near zero (explicitly telling the student what is wrong)

This is the key insight from Hinton's 2015 distillation work.


In [ ]:
# 交互演示：硬标签 vs 软标签，用真实概率分布对比
print("=== 硬标签 vs 软标签 ===")
print()

cities = ["巴黎", "伦敦", "柏林", "罗马", "马德里", "东京", "北京", "悉尼"]
teacher_logits = np.array([5.0, 2.0, 1.0, 0.5, 0.1, -3.0, -4.0, -5.0])

# 硬标签 (SFT): one-hot
hard_labels = np.zeros(len(cities))
hard_labels[0] = 1.0

print("question: 法国的首都是？")
print()
print("硬标签 (SFT):")
for city, prob in zip(cities[:5], hard_labels[:5]):
    bar = "█" * int(prob * 40)
    print(f"  {city}: {prob:.1%} {bar}")
print("  -> Student 只知道「巴黎是对的」")
print()

# 软标签 (distillation): 概率分布
temperature = 3.0
scaled_logits = teacher_logits / temperature
soft_labels = np.exp(scaled_logits) / np.exp(scaled_logits).sum()

print("软标签 (distillation, T=3):")
for city, prob in zip(cities, soft_labels):
    bar = "█" * int(prob * 40)
    print(f"  {city}: {prob:.1%} {bar}")
print("  -> Student 学到了:")
print("     1. 巴黎最对")
print("     2. 伦敦、柏林也是欧洲首都（相似性知识）")
print("     3. 东京、北京概率≈0（完全不相关）")
print()

# 量化信息量差异
hard_entropy = -np.sum(hard_labels * np.log(hard_labels + 1e-10))
soft_entropy = -np.sum(soft_labels * np.log(soft_labels + 1e-10))
print(f"硬标签信息熵: {hard_entropy:.2f} bits")
print(f"软标签信息熵: {soft_entropy:.2f} bits")
print(f"-> 软标签包含约 {soft_entropy:.1f} bits 信息，远多于硬标签的 {hard_entropy:.2f} bits！")

### 2. Method 1: logit distillation (the classic)

Make the student's output distribution match the teacher's output distribution.

Loss:

$$L = (1-lpha) \cdot L_{CE}(S, y) + lpha \cdot T^2 \cdot L_{KL}(S_T, T_T)$$

Where:
- $L_{CE}$: cross entropy against the ground truth (keeps correctness)
- $L_{KL}$: KL divergence between student and teacher distributions (learns dark knowledge)
- $T$: temperature (larger T makes the distribution softer)
- $lpha$: balance weight

Temperature intuition:

```
T=1:  [0.90, 0.05, 0.03, 0.02]  (very sharp)
T=5:  [0.40, 0.25, 0.20, 0.15]  (softer; dark knowledge shows)
T=20: [0.28, 0.26, 0.24, 0.22]  (too soft; close to uniform)
```

Common range: T=3~10.


In [ ]:
# 演示温度对概率分布的影响
print("=== 温度 T 对软标签的影响 ===")
print()

logits = np.array([5.0, 2.0, 1.0, 0.5, 0.1, 0.01, 0.001, 0.0001])
labels = ["巴黎", "伦敦", "柏林", "罗马", "马德里", "维也纳", "布拉格", "华沙"]

for T in [1, 3, 10, 20]:
    scaled = logits / T
    probs = np.exp(scaled) / np.exp(scaled).sum()
    
    print(f"T={T:2d}: ", end="")
    for i in range(5):
        bar = "█" * int(probs[i] * 50)
        print(f"{labels[i]}:{probs[i]:.3f} {bar}  ", end="")
    print()

print()
print("T=1:  几乎只有巴黎 -> 暗知识被掩盖")
print("T=3:  伦敦、柏林也有一定概率 -> 暗知识浮现")
print("T=10: 分布更均匀 -> 暗知识丰富但信号变弱")
print("T=20: 几乎均匀 -> 信息量太少")

### 3. Method 2: data distillation (the most practical)

Logit distillation assumes teacher and student share the **same tokenizer/vocab**.
For LLMs this is often false (GPT-4 vs Qwen, etc.).

**Data distillation bypasses this**: let the teacher generate high-quality training data; then do SFT on the student.

```
Step 1: collect prompts (from your real business use cases)
  ["Write a poem about spring", "Explain quantum mechanics", "Translate: Hello -> Chinese", ...]

Step 2: teacher generates answers
  GPT-4: "Spring is coming..."
  GPT-4: "Quantum mechanics studies..."

Step 3: train student on (prompt, teacher_answer)
  Standard SFT: student imitates teacher's style and quality
```

Pros: no vocab alignment requirement.
Cons: learns "what answers look like", but not the teacher's full distribution.

Advanced tricks:
- multi-turn dialogue distillation
- CoT distillation (teacher provides reasoning traces)
- rejection sampling (generate multiple answers, keep the best)


In [ ]:
# 模拟数据distillationworkflow
print("=== 数据distillationworkflow模拟 ===")
print()

prompts = [
    "解释什么是机器学习",
    "写一首关于秋天的五言诗",
    "Python 中 list 和 tuple 的区别",
]

# 模拟 Teacher (GPT-4) 生成
teacher_responses = [
    "机器学习是人工智能的一个分支，让计算机从数据中学习模式，而不需要显式编程。",
    "秋风扫落叶，霜降百花残。独坐寒窗下，思君衣可单。",
    "list 是可变的（可以增删改），tuple 是不可变的（创建后不能修改）。list 用 []，tuple 用 ()。",
]

print("生成训练数据:")
for i, (prompt, response) in enumerate(zip(prompts, teacher_responses)):
    print(f"\n--- 样本 {i+1} ---")
    print(f"User: {prompt}")
    print(f"Assistant: {response}")

print()
print(f"共生成 {len(prompts)} 条训练数据")
print("Student 在这些数据上做 SFT，学习模仿 Teacher 的风格。")
print()
print("实际项目中，通常需要 1万~10万 条这样的数据。")

### 4. Method 3: feature distillation (advanced)

Not only match output distributions, but also match intermediate representations.

```
Teacher (96 layers):  ... -> Layer 48 -> ... -> Output
                           ^
Student (32 layers):  ... -> Layer 16 -> ... -> Output
  match student Layer 16 to teacher Layer 48 (after projection)
```

Why it can help: intermediate layers encode rich "understanding" signals.

Why it is less common for LLMs:
- closed-source teachers do not expose hidden states
- teacher/student dimensions differ (need projection)
- higher compute and memory

In practice, data distillation is the dominant approach in LLM distillation.


### 5. Practical recipe: distill a 7B model from GPT-4

Below is a concrete end-to-end workflow.


In [ ]:
print("=== Practical：GPT-4 -> 7B distillationworkflow ===")
print()

steps = [
    ("Step 1: 选基座模型", [
        "推荐: Qwen2.5-7B / Llama-3-8B / Mistral-7B",
        "要求: 基座模型本身有一定能力（不能太差）",
        "选 Instruct 版本（已经会遵循指令）",
    ]),
    ("Step 2: 收集 prompts", [
        "来源 1: 你的业务数据（用户真实question）",
        "来源 2: 开源数据集（OpenHermes, ShareGPT, WildChat）",
        "来源 3: 自建——用另一个 LLM 生成多样化 prompt",
        "数量: 至少 5000 条，推荐 5万~10万 条",
    ]),
    ("Step 3: Teacher 生成回答", [
        "用 GPT-4 API 为每个 prompt 生成回答",
        "system prompt: '你是一个有帮助的助手，请详细、准确地回答。'",
        "temperature=0.7（保留一定多样性）",
        "成本: 5万条 × ~500 token/条 ≈ 2500万 token ≈ $250 (GPT-4o)",
    ]),
    ("Step 4: 数据清洗", [
        "去掉太短的回答（<20 token）",
        "去掉包含 '作为 AI' 等拒绝回答的",
        "去掉格式错乱的",
        "去重（相似度 > 0.9 的只保留一条）",
    ]),
    ("Step 5: SFT 训练", [
        "工具: LLaMA-Factory / Axolotl / Firefly",
        "格式: ChatML 或 ShareGPT 格式",
        "超参: lr=2e-5, epochs=3, batch_size=128",
        "硬件: 4×A100 (80G), 训练时间 ~6-12 小时",
    ]),
    ("Step 6: 评测", [
        "用 lm-eval 跑标准 benchmark（见支线 19）",
        "人工评估 100 条业务数据",
        "对比 Student 和 Teacher 的差距",
    ]),
]

for title, details in steps:
    print(title)
    for d in details:
        print(f"  {d}")
    print()

### 6. Distillation vs OPD: when to use which?

| Dimension | Data distillation | OPD (On-Policy Distillation) |
|:---|:---|:---|
| When teacher is used | before training (generate data) | during training (online scoring) |
| Student data | teacher-written answers | student-generated answers |
| Engineering complexity | low (basically SFT) | high (rollouts + online teacher) |
| Exposure bias | yes | no |
| Vocab requirement | none | often needs alignment (or cross-tokenizer distill) |
| Cost | lower (teacher runs once) | higher (teacher runs repeatedly) |
| Best for | fast iteration, limited budget | max performance with strong eng support |

Recommendation: start with data distillation to get a strong baseline quickly.
If it is not enough, consider OPD.


In [ ]:
# distillation效果模拟
print("=== distillation效果对比（模拟） ===")
print()

print("假设 Teacher 是 GPT-4，在 MMLU 上得分 86.4")
print()

models = [
    ("GPT-4 (Teacher)", 86.4, "基线"),
    ("7B 基座 (无distillation)", 64.2, "原始能力"),
    ("7B + 数据distillation", 72.8, "+8.6 分"),
    ("7B + 数据distillation + CoT", 75.1, "+10.9 分"),
    ("7B + OPD", 77.3, "+13.1 分（但成本高 10 倍）"),
]

print(f"{'模型':<25s} {'MMLU':>8s} {'提升':>8s}")
print("-" * 45)
for name, score, note in models:
    improvement = score - 64.2
    print(f"{name:<25s} {score:>6.1f}  {improvement:>+6.1f}  ({note})")

print()
print("Conclusion：数据distillation性价比最高，OPD 效果最好但成本高。")
print("实际项目中，数据distillation + CoT distillation是最常用的组合。")

### 7. Common questions

| Question | Answer |
|:---|:---|
| Can student surpass teacher? | In theory no (upper bound is the teacher), but with extra data the student can outperform on some narrow tasks. |
| What is lost? | creativity, long-tail knowledge, complex reasoning are hardest to distill. |
| How much data? | minimum ~5k, recommended 50k+. Quality > quantity. |
| Different tokenizers? | use data distillation (no vocab alignment). |
| Need RLHF after distill? | depends; if teacher is aligned, student often inherits alignment. |
| Multi-teacher distillation? | yes: e.g., one teacher for reasoning, one for writing, one for multimodal. |


---

## Distillation Summary

1. [ok] Essence: learn teacher distributions (dark knowledge), not only hard labels
2. [ok] Logit distillation: match distributions; usually needs aligned vocab/tokenizer
3. [ok] Data distillation: teacher generates data; student does SFT (most practical)
4. [ok] Feature distillation: match hidden states; strong but expensive and often unavailable
5. [ok] Recipe: base model -> collect prompts -> teacher generate -> clean -> SFT -> eval
6. [ok] Distill vs OPD: data distill is cheaper; OPD is stronger but expensive
7. [ok] CoT distillation: teacher provides reasoning traces; student learns reasoning

One-sentence summary: distillation is a big teacher and a small student; the most practical path is to let GPT-4 write tens of thousands of answers and train the small model to imitate.
